In [39]:
import pandas as pd
import ollama
model = "llama3.2:latest"

In [79]:
# Define the inclusion criteria
prompt = """ Please include this paper if it contains:

As a systematic review expert, evaluate the provided article title {Title} and abstract {Abstract} against the specified inclusion criteria.
   Follow these precise instructions:

   1. Carefully read the article title and abstract.
   2. Assess whether the article meets ALL of the following inclusion criteria:

   a. Study design: exclude meta-analyses or systematic review or narrative review.
   b. Statistical methods: Statistical tests or analytical techniques or modelling should have been used.
   c. Modelling: It should involve statistical modelling or analysis.
   d. Optimization: It should involve optimization or reinforced learning or better algorithms.

  3. Extract the following information:
- Species: Which animal species were studied? (If not specified, write 'unknown')
- Country: In which country the study was conducted? (If not specified, write 'unknown')
- Statistics: Which statistical analysis was used? (If not specified, write 'unknown')
- Optimization: It was considered any kind of optimization? or reinforced learning or better algorithms (if not specified, write 'unknown')
- Strategies: Which are the strategies that were recomended? (If not specified, write 'unknown')
Always output the fields in this order: classification, species, country, statistics, optimization, strategies. All values must be lowercase

4. Provide your assessment using ONLY one of these two responses:
- If ALL criteria are met: true
- If ANY criterion is not met: false

5. Do not include any additional words, punctuation, or explanation in your response.

Example output (use this exact format, one line, comma-separated):
true, pigs, USA, deterministic modelling, optimization, household surveillance   
false, birds, Brazil, systematic review, unknown, unknown, vaccination 

Important:
- Base your assessment solely on the information provided in the title and abstract.
- If any information is unclear or not explicitly stated, write 'unknown'.
- Ensure your response is a single line, comma-separated, in lowercase, without any additional characters.
- If multiple species/countries/statistics are mentioned, list them all.
- You will be penalized for using more than one line or deviating from the format.

Your task is to provide a clear, binary assessment and extract the fields as described above."""

In [80]:
def classify_row(row):
    prompt_text = f"{prompt}\n\nTitle: {row['Title']}\nAbstract: {row['Abstract']}"
    response = ollama.generate(
        model=model,
        prompt=prompt_text
    )
    # Expecting output like: true, dog, usa, logistic regression, 
    parts = [p.strip() for p in response['response'].strip().lower().split(',')]
    # Ensure we always have 5 parts
    while len(parts) < 6:
        parts.append('unknown')
    classification, species, country, statistics, optimization, strategies = parts[:6]
    return pd.Series({
        'classification': 'included' if classification == 'true' else 'not included',
        'species': species,
        'country': country,
        'statistics': statistics,
        'optimization': optimization,
        'strategies': strategies
    })


In [81]:

# Load the CSV file
df = pd.read_csv('pubmed_articles.csv', delimiter=';')

# Apply the classification function to each row and expand the results into new columns
df[['classification', 'species', 'country', 'statistics', 'optimization', 'strategies']] = df.apply(classify_row, axis=1)

# Save the updated dataframe
df.to_csv('classified_papers.csv', index=False)

In [82]:
# Print statistics about the classification
total = len(df)
included = (df['classification'] == 'included').sum()
not_included = (df['classification'] == 'not included').sum()

print(f"Total articles classified: {total}")
print(f"Included: {included}")
print(f"Not included: {not_included}")

# Filter only the included articles
idf = df[df['classification'] == 'included']


# Print a table with the number of articles per country
print("\nNumber of articles per country:")
print(idf['country'].value_counts().to_frame('count'))

# Print a table with the optimization strategies used   
print("\nOptimization:")
print(idf['optimization'].value_counts().to_frame('count'))

# Print a table with the statistical analysisused   
print("\nstatistics:")
print(idf['statistics'].value_counts().to_frame('count'))

# Print a table with the strategies   
print("\nstrategies:")
print(idf['strategies'].value_counts().to_frame('count'))

Total articles classified: 38
Included: 31
Not included: 7

Number of articles per country:
                    count
country                  
unknown                14
usa                     3
china                   2
vietnam                 2
polish republic         1
india                   1
europe/asia             1
russia                  1
poland                  1
lithuania               1
spain                   1
germany                 1
multiple countries      1
south africa            1

Optimization:
              count
optimization       
optimization     30
unknown           1

statistics:
                                                    count
statistics                                               
deterministic modelling                                10
unknown                                                 4
pma qcpr                                                1
moran's i                                               1
qpcr                                

In [75]:
print(idf.head(10))

        PMID       First Author Publication Date  \
0   39937724               Li M         2025 Dec   
1   27938676           Halasa T      2016 Dec 25   
2   37306707             Zeng D         2023 Aug   
3   36700212               Li L             2022   
4   36680090               Li L      2022 Dec 23   
5   26664978          Chenais E             2015   
6   33930624           Hayes BH      2021 Apr 18   
8   38206527             Kimi R      2024 Jan 11   
9   27611939  Giménez-Lirola LG             2016   
10  34260273              Liu H      2021 Sep 20   

                                                Title  \
0   Insights and progress on epidemic characterist...   
1   Control of African swine fever epidemics in in...   
2   Quickly assessing disinfection effectiveness t...   
3   A highly efficient indirect ELISA and monoclon...   
4   The Indirect ELISA and Monoclonal Antibody aga...   
5   African Swine Fever in Uganda: Qualitative Eva...   
6   Mechanistic modelling of

In [77]:
# Print the first 10 included articles: only Title and classification response variables
print(idf[['Title', 'classification', 'statistics']].head(10))

                                                Title classification  \
0   Insights and progress on epidemic characterist...       included   
1   Control of African swine fever epidemics in in...       included   
2   Quickly assessing disinfection effectiveness t...       included   
3   A highly efficient indirect ELISA and monoclon...       included   
4   The Indirect ELISA and Monoclonal Antibody aga...       included   
5   African Swine Fever in Uganda: Qualitative Eva...       included   
6   Mechanistic modelling of African swine fever: ...       included   
8   Spatio-temporal dynamics and distributional tr...       included   
9   Detection of African Swine Fever Virus Antibod...       included   
10  A Semiautomated Luciferase Immunoprecipitation...       included   

                 statistics  
0                   unknown  
1   deterministic modelling  
2              optimization  
3   deterministic modelling  
4   deterministic modelling  
5    household surveillance